In [1]:
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html, Input, Output

df = pd.read_csv("vendas.csv", low_memory=False)
df_g = pd.read_excel("gerentes_lojas.xlsx")

display(df.shape)
display(df.head())


(100100, 8)

,ID_Pedido,Data,Loja,Produto,Preco_Unitario,Qtd,Cliente,Data_Base
0,1,2023-06-08,São Paulo,Mouse Gamer,120.0,1,Cliente_4095,2025-01-01
1,2,2023-03-01,Belo Horizonte,iPhone 14,5500.0,1,Cliente_8750,NaN
2,3,2023-02-25,NaN,"Monitor 27""",1200.0,1,Cliente_14859,NaN
3,4,2024-11-19,RIO DE JANEIRO,Mouse Gamer,120.0,2,Cliente_17343,NaN
4,5,2024-01-27,Rio de Janeiro,Smartphone Samsung,2200.0,1,Cliente_23377,NaN


In [2]:
df_analise = df.drop("Data_Base", axis=1).dropna()

df_analise["Loja"] = df_analise["Loja"].str.strip().str.title()
df_analise["Data"] = pd.to_datetime(df_analise["Data"])

df_analise = df_analise.rename(columns={"Qtd": "Quantidade"})
df_analise = df_analise.drop_duplicates(subset=["ID_Pedido"])

df_g["Loja"] = df_g["Loja"].str.strip().str.title()
df_g["Gerente"] = df_g["Gerente"].str.strip().str.title()
df_g["Meta_Mensal"] = pd.to_numeric(
    df_g["Meta_Mensal"],
    errors="coerce"
)

display(df_g)


,Loja,Gerente,Meta_Mensal
0,São Paulo,Carlos,1790000
1,Rio De Janeiro,Fernanda,1800000
2,Curitiba,Roberto,1750000
3,Belo Horizonte,Juliana,1770000
4,Recife,Marcos,1790000
5,Porto Alegre,Pedro,1770000
6,Salvador,Ana,1800000


In [3]:
df_analise["Faturamento"] = (
    df_analise["Preco_Unitario"] * df_analise["Quantidade"]
)

faturamento_metas = (
    df_analise.assign(
        Ano=df_analise["Data"].dt.year,
        Mes=df_analise["Data"].dt.month
    )
    .groupby(["Loja", "Ano", "Mes"], as_index=False)["Faturamento"]
    .sum()
    .merge(df_g, on="Loja", how="left", validate="many_to_one")
)

faturamento_metas["Atingimento (%)"] = (
    faturamento_metas["Faturamento"] /
    faturamento_metas["Meta_Mensal"] * 100
)

faturamento_metas["Meta atingida"] = (
    faturamento_metas["Atingimento (%)"]
    .ge(100)
    .map({True: "Atingida", False: "Não atingida"})
)

display(faturamento_metas.head())

df_analise["Mes"] = df_analise["Data"].dt.month

faturamento_mes_loja = df_analise.pivot_table(
    index="Loja",
    columns="Mes",
    values="Faturamento",
    aggfunc="sum"
)

nome_mes = {
    1: "Janeiro",
    2: "Fevereiro",
    3: "Março",
    4: "Abril",
    5: "Maio",
    6: "Junho",
    7: "Julho",
    8: "Agosto",
    9: "Setembro",
    10: "Outubro",
    11: "Novembro",
    12: "Dezembro"
}

faturamento_mes_loja = faturamento_mes_loja.rename(columns=nome_mes)

faturamento_mes_loja["Total"] = faturamento_mes_loja.sum(axis=1)
faturamento_mes_loja = faturamento_mes_loja.sort_values(
    "Total",
    ascending=False
)

,Loja,Ano,Mes,Faturamento,Gerente,Meta_Mensal,Atingimento (%),Meta atingida
0,Belo Horizonte,2023,1,1779100.0,Juliana,1770000,100.514124,Atingida
1,Belo Horizonte,2023,2,1832950.0,Juliana,1770000,103.556497,Atingida
2,Belo Horizonte,2023,3,1871600.0,Juliana,1770000,105.740113,Atingida
3,Belo Horizonte,2023,4,1838640.0,Juliana,1770000,103.877966,Atingida
4,Belo Horizonte,2023,5,1997800.0,Juliana,1770000,112.870056,Atingida


In [4]:
# Separa os nomes dos meses, sem incluir a coluna Total
meses = faturamento_mes_loja.columns.drop("Total")

# Transforma a tabela em formato adequado para o Plotly
faturamento_mes_loja = faturamento_mes_loja.reset_index()

tabela_grafico = faturamento_mes_loja.melt(
    id_vars=["Loja"],
    value_vars=meses,
    var_name="Mês",
    value_name="Faturamento"
)

# Mantém os meses na ordem correta, de janeiro a dezembro
tabela_grafico["Mês"] = pd.Categorical(
    tabela_grafico["Mês"],
    categories=meses,
    ordered=True
)

display(tabela_grafico.head())

,Loja,Mês,Faturamento
0,Salvador,Janeiro,3339600.0
1,Rio De Janeiro,Janeiro,3588310.0
2,Recife,Janeiro,3538940.0
3,São Paulo,Janeiro,3593080.0
4,Belo Horizonte,Janeiro,3617770.0


In [5]:
faturamento_mensal_lojas = tabela_grafico.pivot(
    index="Loja",
    columns="Mês",
    values="Faturamento"
)

faturamento_mensal_lojas = faturamento_mensal_lojas.reindex(columns=meses)

variacao_lojas = (
    faturamento_mensal_lojas
    .pct_change(axis=1)
    .iloc[:, -1]
    .mul(100)
    .reset_index()
)

variacao_lojas.columns = ["Loja", "Variação (%)"]

display(variacao_lojas)

,Loja,Variação (%)
0,Belo Horizonte,5.876684
1,Curitiba,8.404539
2,Porto Alegre,1.801563
3,Recife,2.177682
4,Rio De Janeiro,6.308535
5,Salvador,5.963263
6,São Paulo,4.075761


In [6]:
faturamento_por_loja = (
    tabela_grafico
    .groupby("Loja", as_index=False)["Faturamento"]
    .sum()
    .sort_values("Faturamento", ascending=True)
)

display(faturamento_por_loja)

,Loja,Faturamento
1,Curitiba,41121720.0
2,Porto Alegre,41678460.0
0,Belo Horizonte,41714890.0
6,São Paulo,42090690.0
3,Recife,42190390.0
4,Rio De Janeiro,42294720.0
5,Salvador,42300610.0


In [7]:
def formatar_reais(valor):
    return (
        f"R$ {valor:,.2f}"
        .replace(",", "X")
        .replace(".", ",")
        .replace("X", ".")
    )


def formatar_percentual(valor):
    return (
        f"{valor:,.2f}%"
        .replace(",", "X")
        .replace(".", ",")
        .replace("X", ".")
    )


def criar_grafico(dados):
    fig = px.line(
        dados,
        x="Mês",
        y="Faturamento",
        color="Loja",
        title="Evolução do faturamento mensal",
        labels={
            "Mês": "Mês",
            "Faturamento": "Faturamento (R$)"
        }
    )

    fig.update_traces(
        hovertemplate="Mês: %{x}<br>Faturamento: R$ %{y:,.2f}",
        mode="lines+markers",
        line=dict(width=2),
        marker=dict(size=6)
    )

    fig.update_layout(
        yaxis_title="Faturamento (R$)",
        yaxis_tickformat=",.0f",
        separators=",.",
        legend=dict(
            title="Lojas",
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="center",
            x=0.5
        )
    )

    return fig


def criar_grafico_comparativo(dados):
    fig = px.bar(
        dados,
        x="Faturamento",
        y="Loja",
        orientation="h",
        title="Faturamento total por loja",
        labels={
            "Faturamento": "Faturamento (R$)",
            "Loja": "Loja"
        }
    )

    fig.update_traces(
        texttemplate="R$ %{x:,.2f}",
        textposition="outside",
        cliponaxis=False,
        hovertemplate="Loja: %{y}<br>Faturamento: R$ %{x:,.2f}"
    )

    fig.update_layout(
        separators=",.",
        xaxis_title="Faturamento (R$)",
        yaxis_title="",
        showlegend=False,
        margin=dict(r=140)
    )

    return fig


def criar_grafico_variacao(dados):
    dados = dados.sort_values("Variação (%)", ascending=True).copy()
    dados["Variação formatada"] = dados["Variação (%)"].map(formatar_percentual)

    figura = px.bar(
        dados,
        x="Variação (%)",
        y="Loja",
        orientation="h",
        text="Variação formatada",
        title="Variação do faturamento por loja"
    )

    figura.update_traces(
        texttemplate="%{text}",
        textposition="outside",
        cliponaxis=False,
        hovertemplate="Loja: %{y}<br>Variação: %{x:.2f}%"
    )

    figura.update_layout(
        separators=",.",
        xaxis_title="Variação (%)",
        yaxis_title="",
        showlegend=False,
        margin=dict(r=90)
    )

    return figura


In [8]:
opcoes_lojas = [
    {"label": "Todas as lojas", "value": "Todas as lojas"}
]

opcoes_lojas += [
    {"label": loja, "value": loja}
    for loja in sorted(tabela_grafico["Loja"].unique())
]

# Figura inicial do gráfico mensal
fig = criar_grafico(tabela_grafico)

# Figura comparativa: permanece com todas as lojas
fig_comparativo = criar_grafico_comparativo(faturamento_por_loja)

fig_variacao = criar_grafico_variacao(variacao_lojas)

opcoes_anos = [
    {"label": str(ano), "value": int(ano)}
    for ano in sorted(faturamento_metas["Ano"].unique())
]

opcoes_meses = [
    {"label": nome, "value": numero}
    for numero, nome in nome_mes.items()
]

ultimo_periodo = (
    faturamento_metas[["Ano", "Mes"]]
    .drop_duplicates()
    .sort_values(["Ano", "Mes"])
    .iloc[-1]
)

display(opcoes_lojas)

[{'label': 'Todas as lojas', 'value': 'Todas as lojas'},
 {'label': 'Belo Horizonte', 'value': 'Belo Horizonte'},
 {'label': 'Curitiba', 'value': 'Curitiba'},
 {'label': 'Porto Alegre', 'value': 'Porto Alegre'},
 {'label': 'Recife', 'value': 'Recife'},
 {'label': 'Rio De Janeiro', 'value': 'Rio De Janeiro'},
 {'label': 'Salvador', 'value': 'Salvador'},
 {'label': 'São Paulo', 'value': 'São Paulo'}]

In [9]:
app = Dash(__name__)

app.layout = html.Div(
    style={
        "backgroundColor": "#F3F4F6",
        "minHeight": "100vh",
        "padding": "30px",
        "fontFamily": "Arial, sans-serif"
    },
    children=[
        # Título
        html.H1(
            "Análise de faturamento por loja",
            style={
                "color": "#1F2937",
                "fontSize": "30px",
                "marginBottom": "8px"
            }
        ),

        # Subtítulo
        html.P(
            "Acompanhe a evolução mensal do faturamento das lojas.",
            style={
                "color": "#6B7280",
                "fontSize": "15px",
                "marginBottom": "25px"
            }
        ),

        # Cartões de indicadores
        html.Div(
            children=[
                # Faturamento total
                html.Div(
                    children=[
                        html.P("Faturamento total"),
                        html.H2("Calculando...", id="kpi-faturamento")
                    ],
                    style={
                        "backgroundColor": "#FFFFFF",
                        "padding": "20px",
                        "borderRadius": "12px",
                        "flex": "1",
                        "boxShadow": "0 2px 8px rgba(0, 0, 0, 0.08)"
                    }
                ),

                # Loja em destaque
                html.Div(
                    children=[
                        html.P("Loja em destaque"),
                        html.H2("Calculando...", id="kpi-loja")
                    ],
                    style={
                        "backgroundColor": "#FFFFFF",
                        "padding": "20px",
                        "borderRadius": "12px",
                        "flex": "1",
                        "boxShadow": "0 2px 8px rgba(0, 0, 0, 0.08)"
                    }
                ),

                # Melhor mês
                html.Div(
                    children=[
                        html.P("Melhor mês"),
                        html.H2("Calculando...", id="kpi-mes")
                    ],
                    style={
                        "backgroundColor": "#FFFFFF",
                        "padding": "20px",
                        "borderRadius": "12px",
                        "flex": "1",
                        "boxShadow": "0 2px 8px rgba(0, 0, 0, 0.08)"
                    }
                ),

                # Variação mensal
                html.Div(
                    children=[
                        html.P("Variação mensal"),
                        html.H2("Calculando...", id="kpi-variacao")
                    ],
                    style={
                        "backgroundColor": "#FFFFFF",
                        "padding": "20px",
                        "borderRadius": "12px",
                        "flex": "1",
                        "boxShadow": "0 2px 8px rgba(0, 0, 0, 0.08)"
                    }
                )
            ],
            style={
                "display": "flex",
                "gap": "20px",
                "marginBottom": "25px",
                "flexWrap": "wrap"
            }
        ),

        # Seletor de lojas
        html.Div(
            children=[
                html.Label(
                    "Selecione uma loja:",
                    style={
                        "fontWeight": "bold",
                        "color": "#374151",
                        "display": "block",
                        "marginBottom": "10px"
                    }
                ),

                dcc.Dropdown(
                    id="seletor-loja",
                    options=opcoes_lojas,
                    value="Todas as lojas",
                    clearable=False
                )
            ],
            style={
                "marginBottom": "25px"
            }
        ),

        # Gráfico
        html.Div(
            children=[
                dcc.Graph(
                    id="grafico-faturamento",
                    figure=fig
                )
            ],
            style={
                "backgroundColor": "#FFFFFF",
                "padding": "20px",
                "borderRadius": "12px",
                "boxShadow": "0 2px 8px rgba(0, 0, 0, 0.08)"
            }
        ),

        # Gráfico comparativo de faturamento por loja
        html.Div(
            children=[
                dcc.Graph(
                    id="grafico-comparativo",
                    figure=fig_comparativo
                )
            ],
            style={
                "backgroundColor": "#FFFFFF",
                "padding": "20px",
                "borderRadius": "12px",
                "boxShadow": "0 2px 8px rgba(0, 0, 0, 0.08)",
                "marginTop": "20px"
            }
        ),
        
        html.Div(
            children=[
                dcc.Graph(
                    id="grafico-variacao",
                    figure=fig_variacao
                )
            ],
            style={
                "backgroundColor": "#FFFFFF",
                "padding": "20px",
                "borderRadius": "12px",
                "boxShadow": "0 2px 8px rgba(0, 0, 0, 0.08)",
                "marginTop": "20px"
            }
        ),

        # Acompanhamento de metas
html.Div(
    children=[
        html.H3("Acompanhamento de metas"),

        html.Div(
            children=[
                html.Div([
                    html.Label("Selecione o ano"),
                    dcc.Dropdown(
                        id="seletor-ano-meta",
                        options=opcoes_anos,
                        value=int(ultimo_periodo["Ano"]),
                        clearable=False
                    )
                ]),

                html.Div([
                    html.Label("Selecione o mês"),
                    dcc.Dropdown(
                        id="seletor-mes-meta",
                        options=opcoes_meses,
                        value=int(ultimo_periodo["Mes"]),
                        clearable=False
                    )
                ])
            ],
            style={
                "display": "grid",
                "gridTemplateColumns": "1fr 1fr",
                "gap": "20px"
            }
        )
    ],
    style={
        "backgroundColor": "#FFFFFF",
        "padding": "20px",
        "borderRadius": "12px",
        "boxShadow": "0 2px 8px rgba(0, 0, 0, 0.08)",
        "marginTop": "20px"
    }
),

# Resultados do acompanhamento de metas
html.Div(
    children=[
        html.Div(id="resumo-metas"),
        html.Div(id="tabela-metas")
    ],
    style={
        "backgroundColor": "#FFFFFF",
        "padding": "20px",
        "borderRadius": "12px",
        "boxShadow": "0 2px 8px rgba(0, 0, 0, 0.08)",
        "marginTop": "20px"
    }
),
    ]
)


In [10]:
@app.callback(
    Output("grafico-faturamento", "figure"),
    Output("kpi-faturamento", "children"),
    Output("kpi-loja", "children"),
    Output("kpi-mes", "children"),
    Output("kpi-variacao", "children"),
    Output("kpi-variacao", "style"),
    Input("seletor-loja", "value")
)
def atualizar_dashboard(loja_selecionada):

    if loja_selecionada == "Todas as lojas":
        dados_filtrados = tabela_grafico
    else:
        dados_filtrados = tabela_grafico[
            tabela_grafico["Loja"] == loja_selecionada
        ]

    faturamento_total = dados_filtrados["Faturamento"].sum()

    faturamento_por_loja = (
        dados_filtrados.groupby("Loja")["Faturamento"]
        .sum()
        .sort_values(ascending=False)
    )
    melhor_loja = faturamento_por_loja.idxmax()

    faturamento_por_mes = (
        dados_filtrados.groupby("Mês", observed=True)["Faturamento"]
        .sum()
        .sort_values(ascending=False)
    )
    melhor_mes = faturamento_por_mes.idxmax()

    faturamento_mensal_dashboard = (
        dados_filtrados.groupby("Mês", observed=True)["Faturamento"]
        .sum()
        .sort_index()
    )

    if (
        len(faturamento_mensal_dashboard) >= 2
        and faturamento_mensal_dashboard.iloc[-2] != 0
    ):
        ultima_variacao = (
            faturamento_mensal_dashboard.pct_change().iloc[-1] * 100
        )
        variacao_formatada = (
            f"{'+' if ultima_variacao > 0 else ''}"
            f"{formatar_percentual(ultima_variacao)}"
        )

        if ultima_variacao > 0:
            variacao_estilo = {"color": "#15803D"}
        elif ultima_variacao < 0:
            variacao_estilo = {"color": "#DC2626"}
        else:
            variacao_estilo = {"color": "#6B7280"}
    else:
        variacao_formatada = "Sem comparação"
        variacao_estilo = {"color": "#6B7280"}

    figura = criar_grafico(dados_filtrados)

    return (
        figura,
        formatar_reais(faturamento_total),
        melhor_loja,
        melhor_mes,
        variacao_formatada,
        variacao_estilo
    )


@app.callback(
    Output("resumo-metas", "children"),
    Output("tabela-metas", "children"),
    Input("seletor-ano-meta", "value"),
    Input("seletor-mes-meta", "value")
)
def atualizar_metas(ano_selecionado, mes_selecionado):

    dados = faturamento_metas[
        (faturamento_metas["Ano"] == ano_selecionado) &
        (faturamento_metas["Mes"] == mes_selecionado)
    ].sort_values("Faturamento", ascending=False)

    if dados.empty:
        return (
            html.P(
                "Nenhum dado encontrado para o período selecionado.",
                style={"color": "#6B7280", "margin": "0"}
            ),
            None
        )

    faturamento_total = dados["Faturamento"].sum()
    lojas_atingidas = dados["Atingimento (%)"].ge(100).sum()
    total_lojas = len(dados)

    resumo = html.Div(
        children=[
            html.Div(
                children=[
                    html.P(
                        "Faturamento total",
                        style={
                            "color": "#6B7280",
                            "fontSize": "14px",
                            "margin": "0 0 8px"
                        }
                    ),
                    html.H3(
                        formatar_reais(faturamento_total),
                        style={
                            "color": "#1F2937",
                            "fontSize": "22px",
                            "margin": "0"
                        }
                    )
                ],
                style={
                    "backgroundColor": "#F9FAFB",
                    "padding": "16px 20px",
                    "borderRadius": "10px",
                    "flex": "1"
                }
            ),
            html.Div(
                children=[
                    html.P(
                        "Lojas que atingiram a meta",
                        style={
                            "color": "#6B7280",
                            "fontSize": "14px",
                            "margin": "0 0 8px"
                        }
                    ),
                    html.H3(
                        f"{lojas_atingidas} de {total_lojas}",
                        style={
                            "color": "#15803D" if lojas_atingidas == total_lojas else "#1F2937",
                            "fontSize": "22px",
                            "margin": "0"
                        }
                    )
                ],
                style={
                    "backgroundColor": "#F9FAFB",
                    "padding": "16px 20px",
                    "borderRadius": "10px",
                    "flex": "1"
                }
            )
        ],
        style={
            "display": "flex",
            "gap": "16px",
            "flexWrap": "wrap",
            "marginBottom": "20px"
        }
    )

    estilo_cabecalho = {
        "padding": "12px 14px",
        "fontWeight": "600",
        "textAlign": "center",
        "borderBottom": "2px solid #E5E7EB"
    }

    cabecalho = html.Tr(
        [
            html.Th(titulo, style=estilo_cabecalho)
            for titulo in [
                "Loja",
                "Gerente",
                "Faturamento",
                "Meta mensal",
                "Atingimento",
                "Situação"
            ]
        ],
        style={
            "backgroundColor": "#1F2937",
            "color": "#FFFFFF"
        }
    )

    
    estilo_celula = {
        "padding": "11px 14px",
        "borderBottom": "1px solid #E5E7EB"
    }

    linhas = [
        html.Tr(
            [
                html.Td(linha["Loja"], style=estilo_celula),
                html.Td(linha["Gerente"], style=estilo_celula),
                html.Td(formatar_reais(linha["Faturamento"]), style=estilo_celula),
                html.Td(formatar_reais(linha["Meta_Mensal"]), style=estilo_celula),
                html.Td(formatar_percentual(linha["Atingimento (%)"]), style=estilo_celula),
                html.Td(
                    linha["Meta atingida"],
                    style={
                        **estilo_celula,
                        "color": "#15803D" if linha["Atingimento (%)"] >= 100 else "#DC2626",
                        "fontWeight": "600"
                    }
                )
            ],
            style={
                "backgroundColor": "#F9FAFB" if indice % 2 == 0 else "#FFFFFF"
            }
        )
        for indice, (_, linha) in enumerate(dados.iterrows())
    ]

    tabela = html.Div(
        html.Table(
            [html.Thead(cabecalho), html.Tbody(linhas)],
            style={
                "width": "100%",
                "minWidth": "700px",
                "textAlign": "center",
                "borderCollapse": "collapse",
                "fontSize": "14px"
            }
        ),
        style={"width": "100%", "overflowX": "auto"}
    )

    return resumo, tabela


In [11]:
app.run(debug=True, port=8050)